# Lab3: dvc

# Lab desription

## Step 1

In [1]:
!pip freeze | grep -E "dvc|git"

dvc==3.66.1
dvc-data==3.18.3
dvc-http==2.32.0
dvc-objects==5.2.0
dvc-render==1.0.2
dvc-studio-client==0.22.0
dvc-task==0.40.2
gitdb==4.0.12
pygit2==1.19.1


In [2]:
!git init
!dvc init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /home/ychernyshov/git/Safety-GenAI-course/Labs/3/.git/
Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>           

In [3]:
!ls -la

total 2992
drwxrwxr-x  6 ychernyshov ychernyshov    4096 Mar  9 13:42 .
drwxrwxr-x 18 ychernyshov ychernyshov    4096 Feb 27 11:41 ..
drwxrwxr-x  3 ychernyshov ychernyshov    4096 Mar  9 13:42 .dvc
-rw-rw-r--  1 ychernyshov ychernyshov     139 Mar  9 13:42 .dvcignore
-rw-rw-r--  1 ychernyshov ychernyshov 3030354 Mar  9 13:41 dvc.ipynb
drwxrwxr-x  7 ychernyshov ychernyshov    4096 Mar  9 13:42 .git
drwxrwxr-x  2 ychernyshov ychernyshov    4096 Feb 27 11:55 .ipynb_checkpoints
-rw-rw-r--  1 ychernyshov ychernyshov    3205 Mar  9 13:08 requirements.txt
drwxrwxr-x  7 ychernyshov ychernyshov    4096 Feb 27 11:54 venv


In [4]:
!cat .dvcignore

# Add patterns of files dvc should ignore, which could improve
# the performance. Learn more at
# https://dvc.org/doc/user-guide/dvcignore


In [5]:
!echo "dvc.ipynb" >> .dvcignore
!echo ".ipynb_checkpoints" >> .dvcignore
!echo "venv" >> .dvcignore
!echo "requirements.txt" >> .dvcignore

In [14]:
!dvc check-ignore ./*

./dvc.ipynb
./requirements.txt
./venv


In [15]:
!cat .dvcignore

# Add patterns of files dvc should ignore, which could improve
# the performance. Learn more at
# https://dvc.org/doc/user-guide/dvcignore
dvc.ipynb
.ipynb_checkpoints
venv
requirements.txt


In [16]:
!cat .dvc/.gitignore

/config.local
/tmp
/cache


In [17]:
!git ls-files

.dvc/.gitignore
.dvc/config
.dvcignore


In [20]:
!git status

On branch master

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .dvc/.gitignore
	new file:   .dvc/config
	new file:   .dvcignore

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .dvcignore

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.ipynb_checkpoints/
	dvc.ipynb
	requirements.txt
	venv/



In [21]:
!git commit -m "Init DVC project"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'ychernyshov@ychernyshov-VirtualBox.(none)')


!git rm --cached -rf <filename>

## Create a dataset

In [30]:
import os
import numpy as np
import pandas as pd

In [31]:
def generate_clean_data(filename="data/train.csv"):
    np.random.seed(42)
    # 1000 сэмплов, 5 признаков
    X = np.random.rand(1000, 5)
    # Простая логика: если сумма признаков > 2.5, класс 1, иначе 0
    y = (X.sum(axis=1) > 2.5).astype(int)
    
    df = pd.DataFrame(X, columns=[f"feat_{i}" for i in range(5)])
    df['label'] = y
    
    df.to_csv(filename, index=False)
    print(f"Датасет создан: {filename}, строк: {len(df)}")

In [32]:
os.makedirs("data", exist_ok=True)
generate_clean_data()


Датасет создан: data/train.csv, строк: 1000


## Add data to dvc

In [33]:
!dvc add data/train.csv
!git add data/train.csv.dvc .gitignore
!git commit -m "Add clean training data v1.0"

⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/ychernyshov/git/Safety-GenAI-course/Labs/3/.dvc/ca
                                                                                
!
  0%|          |Adding data/train.csv to cache        0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /home/ychernyshov/git/Saf0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00,  9.07file/s]

To track the changes with git, run:

	git add data/train.csv.dvc data/.gitignore

To enable auto staging, run:

	dvc config core.autostage true
fatal: pathspec '.gitignore' did not match any files
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --

## Learn baseline ML model

In [37]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

In [38]:
def train():
    df = pd.read_csv("data/train.csv")
    X = df.drop('label', axis=1)
    y = df['label']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LogisticRegression()
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    
    print(f"Accuracy: {acc:.4f}")
    joblib.dump(model, "model.pkl")
    return acc

In [39]:
train()

Accuracy: 0.9950


0.995

## Data poisoning

In [40]:
def poison_data(filename="data/train.csv"):
    df = pd.read_csv(filename)
    
    # Атака: Инвертируем метки (label) у первых 200 записей
    # Это имитирует внесение шумов или целевое отравление
    df.loc[:200, 'label'] = 1 - df.loc[:200, 'label']
    
    df.to_csv(filename, index=False)
    print("АТАКА ВЫПОЛНЕНА: Данные изменены локально!")



In [41]:
poison_data()

АТАКА ВЫПОЛНЕНА: Данные изменены локально!


## Integrity check with dvc

In [42]:
!dvc status

data/train.csv.dvc:                                                             
	changed outs:
		modified:           data/train.csv


In [43]:
!dvc diff

In [44]:
!md5sum data/train.csv


5c9012de4270bfc0d64a9dc4b0022da4  data/train.csv


## Mitigation

In [46]:
!dvc checkout data/train.csv --force

Building workspace index                              |2.00 [00:00,  150entry/s]
Comparing indexes                                     |3.00 [00:00,  644entry/s]
Applying changes                                      |1.00 [00:00,  68.2file/s]
M       data/train.csv


In [47]:
!dvc status

Data and pipelines are up to date.                                              


## Suspicious commit mitigation

In [48]:
poison_data()

АТАКА ВЫПОЛНЕНА: Данные изменены локально!


In [49]:
!dvc add data/train.csv
!git add data/train.csv.dvc
!git commit -m "Update data (SUSPICIOUS)"

⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/ychernyshov/git/Safety-GenAI-course/Labs/3/.dvc/ca
                                                                                
!
  0%|          |Adding data/train.csv to cache        0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /home/ychernyshov/git/Saf0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 18.02file/s]

To track the changes with git, run:

	git add data/train.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.

In [50]:
# Сравнить текущую версию с предыдущей (HEAD~1)
!dvc diff HEAD~1 HEAD

In [51]:
!git checkout HEAD~1 -- data/train.csv.dvc
!dvc checkout data/train.csv

fatal: invalid reference: HEAD~1
Building workspace index                              |2.00 [00:00, 49.9entry/s]
Comparing indexes                                     |3.00 [00:00,  437entry/s]
Applying changes                                      |0.00 [00:00,     ?file/s]


# dvc commands


даление отслеживаемого файла: dvc remove data/my_dataset.csv.dvc

Удаление файла и его данных:
dvc remove data/my_dataset.csv.dvc
rm data/my_dataset.csv

Очистка кэша DVC (удаление старых версий):
dvc gc -w

Удаление выходов (outputs) стадии dvc.yaml:
dvc remove <stage_name> --outs



# Conclusions

# References

https://github.com/github/gitignore